In [ ]:
#| default_exp quantize.fake_quantize_callback

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import torch
import torch.nn as nn
from collections.abc import Sequence
from dataclasses import replace
from fastai.callback.all import *
from fastcore.basics import store_attr
from torch.nn.utils import parametrize
from fasterai.core.parametrize import _is_parametrized, _master, _unparametrize
from fasterai.core.precision import FAKE_SPEC_ATTR, _type_error
from fasterai.core.schedule import Schedule
from fasterai.quantize.fake_quantizer import (FakeQuantizer, _ActRounder, _check_bits, _fake_quantize,
                                              _qrange, _scale_zero)

## Overview

`FakeQuantizeCallback` trains a model *through* [`FakeQuantizer`](fake_quantizer.html)'s rounding. On
every forward the weights are rounded onto the grid of the width you asked for, the loss is computed on
the rounded model, and the gradient reaches an untouched floating-point copy — the **master** — which the
optimizer keeps training. When the fit ends the rounding is baked in, leaving exactly the model
`FakeQuantizer.quantize_model()` leaves: ordinary modules, floating-point dtype, rounded weights.

That is the one question `FakeQuantizer` cannot answer. Rounding a trained model onto a narrow grid costs
accuracy; letting the weights adapt to the rounding they will be subject to may win some of it back. How
much, on your model, is a number only your fit can produce.

Everything the post-training page says still holds: nothing is packed, no kernel is swapped, and biases
stay in floating point. Two more things this callback deliberately does not do:

- **BatchNorm is untouched** — not folded, not frozen. A deployed flow folds it before quantizing, which
  changes the per-channel weight ranges the rounding sees; fold it yourself first with
  [`BN_Folder`](../misc/bn_folding.html) if you want the model a deployed flow would produce.
- **The activation rounding sits on the module's own output**, before whatever activation function
  follows it. No integer runtime does that — it fuses `conv + relu` and quantizes once, after the
  fusion — so an activation width measured here is a proxy, not the arithmetic a backend will run.

## The mechanism

The weight rounding is a [`torch.nn.utils.parametrize`](../core/parametrize.html) parametrization: the
trained parameter becomes `m.parametrizations.weight.original`, and `m.weight` is that master rounded, on
every access. Three consequences worth knowing:

- **The optimizer needs no rebinding.** `register_parametrization` keeps the *same* `Parameter` object as
  the master, and `remove_parametrizations` hands the same object back, so the optimizer `fit` built
  before the callback ran holds exactly the tensors being trained.
- **The gradient is straight-through.** The forward uses the value `_fake_quantize` writes — the same
  function the post-training path calls, so both round identically — and the backward passes the gradient
  to the master unchanged. Nothing is ever clipped on the weight side: the grid is recomputed from the
  master at every forward, so it always covers it.
- **Anything reading a weight must read the master.** `m.weight` is a rounded snapshot, and a write to it
  is silently discarded. [`Sparsifier`](../sparse/sparsifier.html) and
  [`Criteria`](../core/criteria.html) go through `_master`, which is why sparsification and QAT compose.

Activations are observed the way torch's `MovingAverageMinMaxObserver` observes them: the range moves
towards each batch by `act_averaging` (0.01 by default), rather than being the absolute min and max of
everything ever seen — one outlier batch would otherwise widen the grid permanently, with nothing to
narrow it again. Pass `act_averaging=None` for the absolute min and max. Once a range has been
observed, only training batches move it — the module has to be in training mode — and activations
outside the frozen grid are clipped, which means they get no gradient at all.

| Moment | What happens |
|---|---|
| `before_fit` | builds the `FakeQuantizer`, parametrizes every weight it rounds, hooks every activation |
| every training batch | a scheduled width steps down onto the rung its progress selects |
| every training batch | the forward rounds, the gradient reaches the master, the optimizer moves it |
| `pct_train >= freeze_act_pct` | the activation scales stop moving, and the fit finishes on the grid it will keep |
| `after_fit` | `bake()`: the trained master is rounded into an ordinary weight, and the spec is attached |

`pct_train` is incremented after each batch, so the largest value a fit of `N` steps ever shows is
`(N-1)/N`: a 4-step fit tops out at 0.75 and never reaches the default `freeze_act_pct=0.9`. The scales
are therefore also frozen unconditionally by `bake()`, so a model always leaves a fit on frozen scales.
With `act_bits=None`, or `observer='dynamic'`, there is no scale to freeze and `freeze_act_pct` does
nothing.

In [ ]:
#| export
_DEFAULT_AVERAGING = 0.01  # the constant torch's `MovingAverageMinMaxObserver` uses


class _FakeQuantWeight(nn.Module):
    "Weight parametrization: the module computes its weight by rounding a floating-point master"
    def __init__(self, bits: int, symmetric: bool, qscheme: str, group_size: int | None):
        super().__init__()
        store_attr()

    def forward(self, w):
        q = _fake_quantize(w.detach(), self.bits, self.symmetric, self.qscheme, self.group_size)
        return w + (q - w.detach())  # the value the post-training path writes, passing the gradient on


class _ActObserver:
    "Forward hook rounding a module's output onto a grid it keeps observing, until the scales are frozen"
    def __init__(self, bits: int, symmetric: bool, averaging: float | None):
        store_attr()
        self.frozen, self.enabled = False, True

    def __call__(self, mod, inp, out):
        if not isinstance(out, torch.Tensor): return out
        if (mod.training and not self.frozen) or '_act_scale' not in mod._buffers:
            self._observe(mod, out)
        if not self.enabled: return out  # `FakeQuantizer.calibrate` stands the rounding down to observe
        return torch.fake_quantize_per_tensor_affine(out, mod._act_scale, mod._act_zero_point,
                                                     *_qrange(self.bits, self.symmetric))

    def _observe(self, mod, out) -> None:
        "Move the observed range towards this batch, and rewrite the scales it defines"
        lo, hi = out.detach().amin(), out.detach().amax()
        if '_act_min' in mod._buffers:
            if self.averaging is None:
                lo, hi = torch.minimum(mod._act_min, lo), torch.maximum(mod._act_max, hi)
            else:
                lo, hi = (torch.lerp(mod._act_min, lo, self.averaging),
                          torch.lerp(mod._act_max, hi, self.averaging))
        scale, zero = _scale_zero(lo, hi, self.bits, self.symmetric)
        for name, value in (('_act_min', lo), ('_act_max', hi),
                            ('_act_scale', scale), ('_act_zero_point', zero)):
            mod.register_buffer(name, value, persistent=False)


def _check_ladder(axis: str, schedule, widths, target: int | None) -> tuple[int, ...] | None:
    "Validate one width ladder against the schedule stepping it and the width it must end at"
    if schedule is None and widths is None: return None
    if target is None:
        raise ValueError(f"{axis}_bits=None leaves this axis in floating point, so a ladder has no width "
                         f"to lower: name the width to end at with `{axis}_bits=`, or drop the ladder.")
    if widths is None:
        raise ValueError(f"`{axis}_schedule` has no ladder to step through: pass `{axis}_widths=`, the "
                         f"widths to go down through, coarsest first and ending at {axis}_bits={target}.")
    if isinstance(widths, str) or not isinstance(widths, Sequence):
        raise _type_error(f'{axis}_widths', 'a sequence of widths, coarsest first', widths)
    if schedule is None:
        raise ValueError(f"`{axis}_widths={tuple(widths)}` is never stepped through: pass "
                         f"`{axis}_schedule=`, the fasterai `Schedule` whose progress selects the rung.")
    widths = tuple(widths)
    if len(widths) < 2:
        raise ValueError(f"`{axis}_widths={widths}` steps nothing: pass at least two widths, from the "
                         f"coarsest one down to {axis}_bits={target}, or drop `{axis}_schedule`.")
    for i, w in enumerate(widths):
        if w is None:
            raise ValueError(f"`{axis}_widths[{i}]=None` is not a rung: pass a width in [2, 16] there, "
                             f"and {axis}_bits=None if you meant to leave the axis in floating point.")
        _check_bits(f"{axis}_widths[{i}]", w, f"and only {axis}_bits=None leaves an axis in floating point")
    if widths[-1] != target:
        raise ValueError(f"`{axis}_widths={widths}` must end at the width this rounds to: make its last "
                         f"rung {axis}_bits={target}.")
    if any(a <= b for a, b in zip(widths, widths[1:])):
        raise ValueError(f"`{axis}_widths={widths}` does not go down: pass strictly decreasing widths, "
                         "the coarsest one first.")
    return widths


def _rung(ladder: tuple[int, ...], progress: float) -> int:
    "The width a progress in [0, 1] selects, the rungs sharing it equally"
    return ladder[min(max(int(progress * len(ladder)), 0), len(ladder) - 1)]


class FakeQuantizeCallback(Callback):
    """Quantization-aware training on `FakeQuantizer`'s arithmetic: the fit sees rounded weights, the
    optimizer trains floating-point ones.

    `before_fit` parametrizes every weight this rounds, so the forward computes it from a master the
    gradient reaches; `after_fit` bakes the rounding in and leaves the model `quantize_model()` leaves.
    `self.fake_quantizer` is that quantizer, and its `remove()` gives back the master **as the fit left
    it** — the trained weights, not the ones training started from.

    A fit that raises never reaches `after_fit` and leaves the parametrizations in place, on a model
    `torch.save` refuses: `strip()` takes them off and keeps the floating-point weights, `bake()` keeps
    the rounding.

    A schedule can lower the width during the fit rather than rounding at the target from the first
    batch (`weight_schedule` / `weight_widths`, and their activation twins). Whether that is worth
    doing is not measured here.

    Nothing here folds or freezes BatchNorm, so the weight ranges rounded are not the ones a deployed
    flow would see. The activation rounding sits on each module's own output, before whatever activation
    function follows it, where an integer runtime fuses the two and rounds once."""
    order = 70  # above SaveModelCallback (61), whose after_fit loads a checkpoint into the model

    def __init__(self,
                 weight_bits: int | None = 8,   # Weight width; None leaves them in floating point
                 act_bits: int | None = None,   # Activation width; None leaves them in floating point
                 qscheme: str = 'per_channel',  # Weight scale axis: 'per_tensor', 'per_channel' or 'per_group'
                 symmetric: bool = True,        # Zero-point 0, or an affine offset
                 group_size: int | None = None, # Weights sharing one scale (qscheme='per_group')
                 observer: str = 'static',      # Activation scales: 'static' (observed, then frozen) or 'dynamic'
                 layer_type: type | tuple = (nn.Conv2d, nn.Linear),  # Module types that carry the rounding
                 layer_bits: dict | None = None,      # {layer_name: width} overriding `weight_bits`
                 layer_act_bits: dict | None = None,  # {layer_name: width} overriding `act_bits`
                 freeze_act_pct: float = 0.9,         # Training fraction after which the activation scales stop moving
                 act_averaging: float | None = _DEFAULT_AVERAGING,  # How far the observed range moves towards each batch; None takes its absolute min and max
                 weight_schedule: Schedule | None = None,  # Schedule stepping `weight_widths`; None rounds at `weight_bits` from the first batch
                 weight_widths: Sequence[int] | None = None,  # Weight widths to step down through, coarsest first, ending at `weight_bits`
                 act_schedule: Schedule | None = None,     # Schedule stepping `act_widths`; None rounds at `act_bits` from the first batch
                 act_widths: Sequence[int] | None = None,     # Activation widths to step down through, coarsest first, ending at `act_bits`
                 model: nn.Module | None = None,      # Model to round; None = learn.model
    ):
        "Train through a simulated width, and bake it in when the fit ends"
        store_attr()
        if not 0 <= freeze_act_pct <= 1:
            raise ValueError(f"`freeze_act_pct={freeze_act_pct}` is not a fraction of the training: pass "
                             "a value in [0, 1] (0.9 freezes the scales for the last tenth of the fit).")
        if act_averaging is not None and not 0 < act_averaging <= 1:
            raise ValueError(f"`act_averaging={act_averaging}` is not an averaging constant: pass a value "
                             "in (0, 1], or None to observe the absolute min and max.")
        self._ladders = {'weight': _check_ladder('weight', weight_schedule, weight_widths, weight_bits),
                         'act': _check_ladder('act', act_schedule, act_widths, act_bits)}
        self.current_widths = {'weight': weight_bits, 'act': act_bits}
        self._pinned = {'weight': set(), 'act': set()}
        self.fake_quantizer, self.frozen = None, False
        self._installed, self._baked = False, False

    def before_fit(self) -> None:
        "Build the quantizer, parametrize the weights it rounds, and start observing the activations"
        self._check_alone()
        if self._installed:
            raise RuntimeError("This callback is still installed from a fit that did not finish: call "
                               "strip() for the floating-point weights, or bake() to keep the rounding, "
                               "before fitting again.")
        if self._baked: self.fake_quantizer.remove()  # a second fit resumes from the master the first trained
        self.fake_quantizer = FakeQuantizer(self.model or self.learn.model, self.weight_bits,
                                            self.act_bits, self.qscheme, self.symmetric, self.group_size,
                                            self.observer, self.layer_type, self.layer_bits,
                                            self.layer_act_bits)
        self.frozen, self._baked = False, False
        self._install()
        self.current_widths = {'weight': self.weight_bits, 'act': self.act_bits}
        self._pinned = {'weight': self._pin(self.layer_bits), 'act': self._pin(self.layer_act_bits)}
        for axis, sched, ladder in self._scheduled():
            sched.reset()
            self._set_width(axis, ladder[0])   # the fit starts on the coarsest rung, not on the target
        stepping = ', '.join(f'{axis} through {list(ladder)} bits'
                             for axis, _, ladder in self._scheduled())
        print(f'Training through {self.fake_quantizer.spec.label}, weights {self.qscheme}'
              + (f', stepping {stepping}' if stepping else ''))

    def before_batch(self) -> None:
        "Step the scheduled widths, and freeze the activation scales once the fit is `freeze_act_pct` through"
        if not self.training: return
        self._step_widths()
        if self.pct_train >= self.freeze_act_pct: self._freeze_act()

    def after_fit(self) -> None:
        "Bake the rounding in, unless the model was stripped by hand during the fit"
        if self._installed: self.bake()

    def bake(self) -> nn.Module:
        "Round the trained master into an ordinary weight, and leave the model `quantize_model()` leaves"
        if not self._installed:
            raise RuntimeError("Nothing is installed to bake: run a fit with this callback, which "
                               "installs the rounding and bakes it at the end.")
        fq = self.fake_quantizer
        blind = next((n for n, m in fq._named if isinstance(self._rounder(m), _ActObserver)
                      and '_act_scale' not in m._buffers), None)
        if blind is not None:   # refuse before a single weight is written, as `_check_model` does
            raise RuntimeError(f"'{blind}' never saw a training batch, so nothing observed its activation "
                               "range: fit at least one batch, or leave its activations in floating point "
                               "with layer_act_bits.")
        self._freeze_act()  # a threshold a short fit never reached must not leave the scales moving
        for axis, _, ladder in self._scheduled():
            self._set_width(axis, ladder[-1])  # a fit cut short still bakes the width the spec names
        for _, m in fq._named:
            if not _is_parametrized(m): continue
            m.register_buffer('_fp_weight', _master(m).detach().clone(), persistent=False)
            _unparametrize(m, leave_parametrized=True)
        for m, (handle, rounder) in list(fq._act_hooks.items()):
            for name in ('_act_min', '_act_max'):
                if name in m._buffers: delattr(m, name)
            if not isinstance(rounder, _ActObserver): continue
            handle.remove()  # the scales are frozen: from here the post-training rounder reads them
            ptq = _ActRounder(rounder.bits, rounder.symmetric, False)
            fq._act_hooks[m] = (m.register_forward_hook(ptq), ptq)
        fq._calibrated, fq._quantized = True, True
        setattr(fq.model, FAKE_SPEC_ATTR, replace(fq.spec, trained=True,
                                                  weight_widths=self._ladders['weight'],
                                                  act_widths=self._ladders['act']))
        self._installed, self._baked = False, True
        print(f'Baked in {fq.spec.label}: the model holds its rounded weights, and '
              'fake_quantizer.remove() gives the trained floating-point ones back')
        return fq.model

    def strip(self) -> nn.Module:
        "Drop the rounding and keep the floating-point master — the way back from a fit that raised"
        if self.fake_quantizer is None:
            raise RuntimeError("There is nothing to strip: this callback has not started a fit yet.")
        if self._baked:
            raise RuntimeError("This fit is already baked: call fake_quantizer.remove() for the "
                               "floating-point weights it trained.")
        fq = self.fake_quantizer
        for handle, _ in fq._act_hooks.values(): handle.remove()
        fq._act_hooks = {}
        for _, m in fq._named:
            _unparametrize(m)
            for name in ('_act_min', '_act_max', '_act_scale', '_act_zero_point'):
                if name in m._buffers: delattr(m, name)
        self._installed = False
        return fq.model

    def _install(self) -> None:
        "Parametrize every weight this rounds, and hook every activation it observes"
        fq = self.fake_quantizer
        already = next((n for n, m in fq._named if _is_parametrized(m)), None)
        if already is not None:
            raise RuntimeError(f"'{already}' already computes its weight from a parametrization: remove "
                               "it before training through a simulated width.")
        for _, m in fq._named:
            bits = fq._weight_map[m]
            if bits is not None:
                parametrize.register_parametrization(
                    m, 'weight', _FakeQuantWeight(bits, fq.symmetric, fq.qscheme, fq.group_size))
            act_bits = fq._act_map[m]
            if act_bits is not None:
                rounder = (_ActRounder(act_bits, fq.symmetric, True) if fq.observer == 'dynamic'
                           else _ActObserver(act_bits, fq.symmetric, self.act_averaging))
                fq._act_hooks[m] = (m.register_forward_hook(rounder), rounder)
        self._installed = True

    def _rounder(self, m: nn.Module):
        "The hook rounding this module's output, when it rounds one"
        hook = self.fake_quantizer._act_hooks.get(m)
        return hook[1] if hook else None

    def _freeze_act(self) -> None:
        "Stop the observers moving: from here the fit trains on the scales the model will keep"
        for _, rounder in self.fake_quantizer._act_hooks.values():
            if isinstance(rounder, _ActObserver): rounder.frozen = True
        self.frozen = True

    def _scheduled(self):
        "One (axis, schedule, ladder) triple per scheduled axis, and nothing when no width is scheduled"
        for axis, sched in (('weight', self.weight_schedule), ('act', self.act_schedule)):
            if self._ladders[axis] is not None: yield axis, sched, self._ladders[axis]

    def _pin(self, overrides: dict | None) -> set:
        "The modules a per-layer dict named: their width is the one it gave them, all fit"
        by_name = dict(self.fake_quantizer._named)
        return {by_name.get(k) if isinstance(k, str) else k for k in (overrides or {})} - {None}

    @property
    def _final_batch(self) -> bool:
        "Whether this is the fit's last training batch, whose rung the bake keeps"
        return self.epoch == self.n_epoch - 1 and self.iter == self.n_iter - 1

    def _step_widths(self) -> None:
        "Lower each scheduled axis onto the rung its progress selects"
        for axis, sched, ladder in self._scheduled():
            # the rung decides, not `Schedule.changed`, so a schedule shared with another callback that
            # does read it is left exactly as that one expects to find it
            progress = sched.progress(round(self.pct_train, 3))
            # `pct_train` never reaches 1, so the last batch takes the final rung whatever the arithmetic
            bits = ladder[-1] if self._final_batch else _rung(ladder, progress)
            if bits != self.current_widths[axis]: self._set_width(axis, bits)

    def _set_width(self, axis: str, bits: int) -> None:
        "Write one rung onto every rounder the ladder of `axis` moves, and remember it"
        if axis == 'weight': self._set_weight_width(bits)
        else: self._set_act_width(bits)
        self.current_widths[axis] = bits

    def _set_weight_width(self, bits: int) -> None:
        "Round to `bits` every weight on the ladder — the ones `layer_bits` named are not on it"
        for _, m in self.fake_quantizer._named:
            if m in self._pinned['weight'] or not _is_parametrized(m): continue
            for p in m.parametrizations.weight:
                if isinstance(p, _FakeQuantWeight): p.bits = bits

    def _set_act_width(self, bits: int) -> None:
        "Round to `bits` every activation on the ladder, rewriting the scales its observed range defines"
        for m, (_, rounder) in self.fake_quantizer._act_hooks.items():
            if m in self._pinned['act']: continue
            rounder.bits = bits
            # the observed range is width-independent; only the scale it defines has to be rewritten
            if '_act_min' not in m._buffers: continue
            scale, zero = _scale_zero(m._act_min, m._act_max, bits, self.symmetric)
            for name, value in (('_act_scale', scale), ('_act_zero_point', zero)):
                m.register_buffer(name, value, persistent=False)

    def _check_alone(self) -> None:
        "Refuse a fit that also carries `QuantizeCallback`, which swaps the model this one parametrizes"
        from fasterai.quantize.quantize_callback import QuantizeCallback
        if any(isinstance(cb, QuantizeCallback) for cb in self.learn.cbs):
            raise ValueError("QuantizeCallback and FakeQuantizeCallback both rewrite the model being "
                             "trained, and the first swaps out what the second parametrizes: keep one.")

In [ ]:
show_doc(FakeQuantizeCallback)

The arguments are [`FakeQuantizer`](fake_quantizer.html)'s, spelled the same way and validated by the
same code — the callback builds one at the start of the fit and exposes it as `self.fake_quantizer`, so a
width, an axis or a `group_size` this grammar cannot honor is refused there, with the same sentence, and
before a single weight is touched. The others are the callback's own, and are checked at construction:

- `freeze_act_pct`: how far into the fit the activation scales stop moving. A no-op when there are no
  static activation scales, i.e. with `act_bits=None` or `observer='dynamic'`.
- `act_averaging`: how far the observed range moves towards each batch, `0.01` by default. `None` takes
  the absolute min and max instead, which one outlier batch widens for good.
- `weight_schedule` / `weight_widths` and `act_schedule` / `act_widths`: the width ladder below. Without
  them the rounding is applied at the width you asked for from the first batch, the way `prepare_qat_fx`
  does.
- `model`: the model to round, when it is not `learn.model`.

In [ ]:
show_doc(FakeQuantizeCallback.bake)

In [ ]:
show_doc(FakeQuantizeCallback.strip)

---

## Usage

```python
from fasterai.quantize.fake_quantize_callback import FakeQuantizeCallback

cb = FakeQuantizeCallback(weight_bits=4)            # weight-only QAT at 4 bits, per channel
learn.fit(5, cbs=[cb])

cb = FakeQuantizeCallback(8, 8, freeze_act_pct=0.9)  # weights and activations, scales frozen at 90%
learn.fit(5, cbs=[cb])
```

Bind the callback: `fit` takes it off the learner when it returns, and everything below is called on
it.

After the fit `learn.model` is an ordinary model holding its rounded weights, tagged with the precision
it trained through:

```python
from fasterai.core.precision import fake_quant_spec

fake_quant_spec(learn.model)          # FakeQuantSpec(..., trained=True)
learn.validate()                      # what the trained width costs on your metric
cb.fake_quantizer.remove()            # the TRAINED floating-point weights back
```

`trained=True` is the one thing a spec from this callback says that a post-training one does not: the
model was fitted through the rounding rather than rounded after the fact.

A fit that raises stops before `after_fit`, so the parametrizations are still in place and
`torch.save(learn.model)` refuses to serialize the model. Either finish the job or undo it:

```python
cb.bake()    # keep the rounding: an ordinary model at the width it was training through
cb.strip()   # drop it: the floating-point weights, as the interrupted fit left them
```

### The width ladder

The fit can also *start* wide and step down to the width you asked for, one axis at a time. A ladder is
two arguments: a [`Schedule`](../core/schedules.html), whose progress in [0, 1] selects the rung, and at
least two widths to step through — coarsest first, ending at the width the callback rounds to.

```python
from fasterai.core.schedule import lin, one_shot

# weights start 16-bit and reach the 4 bits `weight_bits` names
learn.fit(5, cbs=[FakeQuantizeCallback(weight_bits=4, weight_schedule=lin, weight_widths=(16, 8, 4))])

# the two axes are independent: 8-bit weights throughout, activations dropped to 4 halfway
learn.fit(5, cbs=[FakeQuantizeCallback(8, 4, act_schedule=one_shot, act_widths=(8, 4))])
```

The rungs share the schedule's progress equally: with three rungs, progress below 1/3 selects the first,
below 2/3 the second, and the rest the last. `cb.current_widths` is the pair the model is rounding at
right now. Three things the mechanism guarantees:

- **The last training batch is always on the final rung**, whatever the arithmetic says. `pct_train` is
  incremented after each batch, so it never reaches 1 — a 4-step fit tops out at 0.75 — and a ladder that
  waited for it would bake a width the spec does not name. `bake()` sets the final rung too, so a fit cut
  short still leaves the model at `weight_bits` / `act_bits`.
- **Changing a width re-observes nothing.** On the activation side the observed range is
  width-independent — only the `scale = range / qmax` it defines moves with the width, and that is
  rewritten from the range the observer already holds. On the weight side there is nothing to redo at
  all: the master stays in floating point and every forward rounds a copy of it, so a rung change is
  simply what the next forward reads.
- **The schedule is only read.** The rung, not `Schedule.changed`, decides whether anything is written,
  and `after_step()` is never called — so handing the same schedule object to a
  [`SparsifyCallback`](../sparse/sparsify_callback.html) in the same fit leaves that callback's own
  reading of it intact.

A layer whose width was named by `layer_bits` or `layer_act_bits` keeps that width for the whole fit —
the ladder moves the layers you did not name.

Each axis refuses on its own, at construction:

| Request | Refusal |
|---|---|
| `weight_schedule=lin` with no `weight_widths` | a schedule has no ladder to step through |
| `weight_widths=(16, 8, 4)` with no `weight_schedule` | a ladder nothing steps through |
| `weight_bits=4, weight_widths=(16, 8)` | the last rung must be the width this rounds to |
| `weight_widths=(8, 8, 4)` or `(4, 8)` | a ladder goes down, strictly |
| `weight_widths=(4,)` | a single rung steps nothing |
| `weight_widths=(17, 8, 4)` | a rung outside the widths this engine rounds to ([2, 16]) |
| `weight_widths=8` | a ladder is a sequence of widths, not one |
| `weight_bits=None` with a `weight_schedule` | an axis left in floating point has no width to lower |

`FakeQuantSpec` records the ladder next to the width the model ended at, so `weight_widths=(16, 8, 4)`
with `weight_bits=4` reads back as `FakeQuantSpec(weight_bits=4, ..., weight_widths=(16, 8, 4))` — a
model that arrived at 4 bits through a ladder is not read as one trained at 4 bits throughout. The field
is the ladder the fit was *asked* to step down: a fit short enough to skip a rung, or one that raised
before reaching one, still records the whole ladder.

**Whether stepping the width down is worth doing is not measured here.** The ladder is a mechanism this
callback offers; which schedule, which rungs, and how any of it compares to training at the target width
from the first batch are questions for a separate experiment, on your model and your data.

### What it costs

Rounding on every forward is not free, and nothing here claims otherwise. One architecture, one machine,
one shape:

| ResNet-18, 10 classes, batch 32 of 3x64x64, CPU | step time | peak resident memory |
|---|---|---|
| plain fit | 109 ms | 1.15 GiB |
| `FakeQuantizeCallback(8, 8)` | 128 ms | 1.27 GiB |

*Measured on one machine: an Intel i9-14900KS with `torch.set_num_threads(4)`, torch 2.9.1, 12 training
steps per run, six runs per arm in fresh processes, alternating between the arms; each figure is the
median of the per-run medians (step time) and of the per-run high-water marks (`ru_maxrss`). That is
1.17x the step time and +120 MiB here.* Those are the numbers this fixture produced, not a target: both
move with the architecture, the batch size and the machine, so measure your own if the budget matters.

The master doubles nothing that was not already there — it *is* the parameter, and the rounded weight is
the extra tensor — while the activation hooks add one rounded copy of every feature map they round.

### With `Sparsifier`

The two compose in one fit, in either order: the mask is applied to the master, and a weight that is
exactly 0 rounds to exactly 0 on a symmetric grid and on an affine one alike, so sparsity survives the
rounding.

```python
learn.fit(5, cbs=[SparsifyCallback(0.5, 'weight', 'local', large_final, one_cycle),
                  FakeQuantizeCallback(weight_bits=8)])
```

`Sparsifier` scores, masks, snapshots and rewinds the master, so `print_sparsity()` reports the mask
rather than the rounding, and a winning ticket is saved as floating-point weights. Structured pruning is
the exception: [`Pruner`](../prune/pruner.html) refuses a parametrized model, because torch-pruning
rewrites the modules it traces. Bake or strip before pruning.

---

## See Also

- [FakeQuantizeCallback tutorial](../tutorials/quantize/fake_quantize_callback.html) - One measured run of the width ladder against training at the target width
- [FakeQuantizer](fake_quantizer.html) - The same arithmetic, applied after training
- [Parametrize](../core/parametrize.html) - The master weight behind a parametrized module
- [QuantizeCallback](quantize_callback.html) - QAT that produces a model running at reduced precision
- [Precision](../core/precision.html) - `FakeQuantSpec`, and what `trained` means on one
- [Schedules](../core/schedules.html) - The `Schedule` a width ladder reads its progress from
- [BN_Folder](../misc/bn_folding.html) - Fold BatchNorm before training through a width
- [SparsifyCallback](../sparse/sparsify_callback.html) - Sparsify in the same fit

Tests live in `nbs/tests/test_fake_quantize_callback.ipynb`.